# E00_Bandit — многорукий бандит: ε-greedy, UCB1, Thompson Sampling

**Урок:** 1.7 курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** три правила разведки на среде с одним состоянием.
**Ожидаемый результат:** все три находят лучшую руку; `Eval/Mean Reward` → **0.80**.
**Время обучения:** ~40 секунд на стратегию (`QUICK_RUN = True` — около 10 секунд).

## Зачем нужен пример без состояний

Бандит — это MDP, из которого убрали всё, кроме одного: **exploration vs
exploitation**. Состояние одно, динамики нет, эпизод длится один шаг. Нет ни
дисконтирования, ни бутстрэппинга, ни «следующего состояния» — а значит, всё,
что здесь происходит, происходит **только** из-за правила выбора действия.

Это делает `E00` идеальным местом, чтобы увидеть разницу между тремя способами
разведки в чистом виде: на любой более сложной среде она смешалась бы
с качеством оценки ценности.

Среда: 5 рук с бернуллиевскими наградами, вероятности
`[0.20, 0.35, 0.50, 0.65, 0.80]` (`ENV_SPEC.md`, §5). Оптимум — **0.80**.

## Требования

```powershell
python scripts\build_env.py E00_Bandit
```


## 1. Окружение

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")


python        3.10.11
torch         2.2.1+cu121  CUDA 12.1
mlagents_envs 1.2.0.dev0
устройство    cuda
сид           0


## 2. Конфигурация

Все гиперпараметры в одном месте. Разделение то же, что и в остальных
примерах: `BanditConfig` — свойства **метода** (правило выбора и его
параметры), `BanditTrainConfig` — свойства **процесса обучения** (бюджет,
частота оценки).

Обратите внимание на `EPSILON`: расписание разведки задаётся снаружи и
используется **только** стратегией `eps_greedy`. У UCB и Thompson разведка
встроена в само правило выбора — отдельного параметра у неё нет. Это и есть
их главное практическое преимущество: нечего настраивать.


In [2]:
QUICK_RUN = True   # True — быстрая проверка (~10 с на стратегию); False — полный прогон (~40 с)

from labrl.algos.bandits import BanditConfig
from labrl.train.bandit import BanditTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 1_000 if QUICK_RUN else 4_000

TRAIN_CFG = BanditTrainConfig(
    total_steps=TOTAL_STEPS,
    eval_every_steps=max(1, TOTAL_STEPS // 8),
    eval_episodes=200,        # награда бернуллиевская: 200 нажатий дают ст. ошибку ~0.03
    success_threshold=0.0,    # успех = нажатие принесло награду
)

# Расписание НЕ сжимается вместе с бюджетом: сокращённый прогон должен быть
# началом полного, а не другим экспериментом (T-10 в docs/07_TROUBLESHOOTING.md).
EPSILON = LinearSchedule(start=1.0, end=0.05, decay_steps=2_000)

STRATEGIES = {
    "eps_greedy": BanditConfig(strategy="eps_greedy"),
    "ucb":        BanditConfig(strategy="ucb", ucb_c=1.414),   # классическое c = √2
    "thompson":   BanditConfig(strategy="thompson", prior_alpha=1.0, prior_beta=1.0),
}

# Истинные вероятности рук из ENV_SPEC.md. В настоящей задаче они неизвестны;
# здесь нужны только для диагностического графика сожаления.
ARM_PROBABILITIES = np.array([0.20, 0.35, 0.50, 0.65, 0.80])

print(f"бюджет: {TOTAL_STEPS} шагов сбора на стратегию, оценка каждые {TRAIN_CFG.eval_every_steps}")
print(f"оптимум среды: {ARM_PROBABILITIES.max():.2f} (рука {ARM_PROBABILITIES.argmax()})")


бюджет: 1000 шагов сбора на стратегию, оценка каждые 125
оптимум среды: 0.80 (рука 4)


## 3. Подключение среды

In [3]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E00_Bandit"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")


behavior: E00_Bandit?team=0
наблюдения (1 сенсоров):
  obs_0: shape=(1,)  имя сенсора: 'VectorSensor_size1'
действия: непрерывных 0, дискретные ветки (5,)

арен (параллельных сред): 8


## 4. Инспекция пространств

`assert` сверяет фактическую среду с `ENV_SPEC.md`. Расхождение обязано падать
здесь, а не проявляться загадочными результатами позже.

Отдельно проверяется, что наблюдение равно **единице**. Это не формальность:
таблица оценок экспортируется в ONNX линейным слоем без смещения, и при
наблюдении 0.0 слой выдал бы нули — модель в Unity всегда выбирала бы руку 0.


In [4]:
OBS_DIM = vec.single_obs_dim
NUM_ARMS = int(vec.action_spec.discrete_branches[0])

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim:           {OBS_DIM}")
print(f"дискретные ветки:  {tuple(vec.action_spec.discrete_branches)}")
print(f"непрерывных:       {vec.action_spec.continuous_size}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=1; discrete_branches=5; continuous_size=0; max_step=1 -->
assert OBS_DIM == 1, f"obs_dim {OBS_DIM} != 1 (ENV_SPEC.md)"
assert NUM_ARMS == 5, f"рук {NUM_ARMS} != 5 (ENV_SPEC.md)"
assert vec.action_spec.continuous_size == 0
assert np.allclose(obs[0], 1.0), f"наблюдение обязано быть константой 1.0, получено {np.unique(obs[0])}"
print("\nразмерности совпадают с ENV_SPEC.md")


obs_shapes:        [(1,)]
obs_dim:           1
дискретные ветки:  (5,)
непрерывных:       0

размерности совпадают с ENV_SPEC.md


## 5. Модель ценности

Нейросети здесь нет, и это не упрощение. У бандита **одно** состояние, поэтому
ценность действия — просто число, а вся «модель» — вектор из K чисел:

$$Q_k = \frac{S_k}{N_k} = \text{среднее наград, полученных рукой } k$$

Оценка ведётся инкрементально, без хранения истории:

$$Q_k \leftarrow Q_k + \frac{1}{N_k}\,(r - Q_k)$$

Шаг $1/N_k$ — это в точности несмещённое выборочное среднее, а не эвристика.

Что здесь можно менять: `initial_value` (оптимистичная инициализация сама
по себе стимулирует разведку — попробуйте 1.0 при ε = 0), `ucb_c` (ширина
разведки UCB), параметры априорного Beta для Thompson.


In [5]:
from labrl.algos.bandits import Bandit

demo = Bandit(num_actions=NUM_ARMS, cfg=STRATEGIES["ucb"])
print(f"состояние алгоритма: {demo.values.shape[0]} оценок, {demo.counts.shape[0]} счётчиков")
print(f"обучаемых параметров нейросети: 0")
print(f"размер таблицы, уезжающей в ONNX: (1, {NUM_ARMS})")


состояние алгоритма: 5 оценок, 5 счётчиков
обучаемых параметров нейросети: 0
размер таблицы, уезжающей в ONNX: (1, 5)


## 6. Теория и математика метода

Все три стратегии оценивают ценность рук одинаково (формула выше) и
различаются **только** правилом выбора.

**ε-greedy.** С вероятностью $\varepsilon$ — случайная рука, иначе
$\arg\max_k Q_k$. Разведка равномерна: метод одинаково часто перепроверяет
и почти-лучшую руку, и заведомо провальную. Простейший метод и худший из трёх
по сожалению.

**UCB1** (Auer et al., 2002) — «оптимизм в условиях неопределённости»:

$$\text{UCB}_k = Q_k + c\,\sqrt{\frac{\ln t}{N_k}}$$

Второе слагаемое — ширина доверительного интервала. Оно велико для рук,
испробованных редко, и стягивается по мере накопления данных. Разведка
направлена туда, где оценка **неточна**, а не куда попало.

**Thompson Sampling** (Thompson, 1933) — байесовский подход. Для
бернуллиевских наград сопряжённое априорное — Beta, поэтому апостериорное
считается точно:

$$\theta_k \sim \mathrm{Beta}(\alpha_0 + \text{успехи}_k,\; \beta_0 + \text{промахи}_k),
\qquad a = \arg\max_k \theta_k$$

Чем меньше данных о руке, тем шире её распределение и тем чаще она случайно
выигрывает розыгрыш — разведка возникает **сама**, без отдельного параметра.

Ниже — исходный код правила выбора и обновления, тот самый, что выполняется
при обучении.


In [6]:
import inspect

print(inspect.getsource(Bandit.act))
print(inspect.getsource(Bandit.update))


    def act(self, batch_size: int, epsilon: float, rng: np.random.Generator) -> np.ndarray:
        """Выбирает руку для каждого из `batch_size` слотов.

        Args:
            batch_size: сколько арен ждут действия.
            epsilon: вероятность случайной руки; используется только
                стратегией ``eps_greedy``, остальные её игнорируют.
            rng: генератор — передаётся явно ради воспроизводимости.

        Returns:
            ``(batch_size,)`` индексы рук.
        """
        if self.cfg.strategy == "eps_greedy":
            greedy = np.full(batch_size, int(np.argmax(self.values)), dtype=np.int64)
            if epsilon <= 0.0:
                return greedy
            explore = rng.random(batch_size) < epsilon
            random_arms = rng.integers(0, self.num_actions, size=batch_size)
            return np.where(explore, random_arms, greedy)

        if self.cfg.strategy == "ucb":
            # Правило детерминировано, поэтому все слоты шага выбирают одну
  

## 7. Цикл обучения

Цикл импортируется из `labrl.train.bandit`, а не копируется сюда: иначе
учебная и рабочая версии со временем разойдутся.

Три стратегии обучаются **последовательно в одной и той же среде**. Среда
не пересоздаётся: вероятности рук фиксированы ТЗ и не зависят ни от сида,
ни от момента запуска, поэтому условия у всех трёх одинаковы.


In [7]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.bandit import train_bandit

results = {}

for name, algo_cfg in STRATEGIES.items():
    algo = Bandit(num_actions=NUM_ARMS, cfg=algo_cfg)
    run = create_run_dir(ENV_ID, name, SEED)
    run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
    logger = TBLogger(run.tb)

    print(f"\n=== {name} === каталог прогона: {run.root.name}")
    result = train_bandit(
        vec=vec, algo=algo, epsilon_schedule=EPSILON, logger=logger,
        cfg=TRAIN_CFG, seed=SEED, arm_probabilities=ARM_PROBABILITIES,
        on_eval=lambda step, r, s: print(f"  шаг {step:>5}: награда {r:+.4f}, успех {s:.0%}"),
    )

    missing = logger.missing_required_tags()
    logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "strategy": name,
                                                 "missing_required_tags": list(missing)})
    logger.close()
    results[name] = (algo, result)
    print(f"  нажатий: {result.pulls}, время: {result.wall_time:.1f} с, "
          f"теги без данных: {missing or 'нет'}")



=== eps_greedy === каталог прогона: 20260815-200347_seed0


  шаг   125: награда +0.7600, успех 76%


  шаг   250: награда +0.8100, успех 81%


  шаг   375: награда +0.8000, успех 80%


  шаг   500: награда +0.7950, успех 80%


...

  шаг   500: награда +0.7950, успех 80%


  шаг   625: награда +0.7650, успех 76%


  шаг   750: награда +0.8000, успех 80%


  шаг   875: награда +0.8050, успех 80%


  шаг  1000: награда +0.7500, успех 75%
  нажатий: 8000, время: 12.4 с, теги без данных: нет


## 8. Оценка и сравнение стратегий

Главный график — **не** награда, а **сожаление** (regret): разница между
наградой, которую дала бы лучшая рука, и наградой выбранной. Награда
бернуллиевская и потому шумная; сожаление — детерминированная функция выбора,
и на нём разница между стратегиями видна сразу.

Обратите внимание: итоговая жадная политика у всех трёх, скорее всего,
**одинакова** — все находят руку 4. Различаются они тем, **сколько стоила**
дорога к ней.


In [8]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

for name, (algo, result) in results.items():
    steps = [s for s, _, _ in result.eval_history]
    rewards = [r for _, r, _ in result.eval_history]
    axes[0].plot(steps, rewards, marker="o", label=name)
    axes[1].bar(np.arange(NUM_ARMS) + 0.25 * list(results).index(name) - 0.25,
                algo.counts / algo.counts.sum(), width=0.25, label=name)

axes[0].axhline(ARM_PROBABILITIES.max(), color="k", ls="--", lw=1, label="оптимум")
axes[0].set_title("Eval/Mean Reward (жадная политика)")
axes[0].set_xlabel("шаг сбора"); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].set_title("Доля нажатий по рукам")
axes[1].set_xlabel("рука"); axes[1].set_xticks(np.arange(NUM_ARMS)); axes[1].grid(alpha=0.3)
axes[1].legend()

names = list(results)
regrets = [results[n][0].regret_per_pull(ARM_PROBABILITIES) for n in names]
axes[2].bar(names, regrets, color=["#c0504d", "#4f81bd", "#9bbb59"])
axes[2].set_title("Сожаление на одно нажатие (меньше — лучше)")
axes[2].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"{'стратегия':<12} {'оценка':>8} {'успех':>7} {'сожаление':>11}  оценки рук")
for name, (algo, result) in results.items():
    print(f"{name:<12} {result.last_eval_reward:>8.4f} {result.last_eval_success:>6.0%} "
          f"{algo.regret_per_pull(ARM_PROBABILITIES):>11.4f}  "
          f"{np.array2string(algo.values, precision=2, floatmode='fixed')}")

THRESHOLD = 0.70   # блок success_criteria конфига
for name, (_, result) in results.items():
    assert result.last_eval_reward >= THRESHOLD, (
        f"{name}: не достигнут критерий приёмки: {result.last_eval_reward:.4f} < {THRESHOLD}")
print(f"\nвсе стратегии прошли порог {THRESHOLD}")


стратегия      оценка   успех   сожаление  оценки рук
eps_greedy     0.7850    78%      0.2308  [0.19 0.35 0.50 0.66 0.80]
ucb            0.8250    82%      0.0204  [0.23 0.35 0.52 0.60 0.80]
thompson       0.7500    75%      0.0043  [0.00 0.39 0.41 0.67 0.81]

все стратегии прошли порог 0.7


C:\Users\-\AppData\Local\Temp\ipykernel_12448\3311184208.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Экспорт в ONNX и верификация

`policy_module()` возвращает таблицу оценок, обёрнутую линейным слоем без
смещения. При наблюдении-константе 1.0 слой выдаёт ровно вектор оценок,
а `argmax` от него — ту же жадную политику, которую мы только что измерили.
Никакого приближения: в Unity уезжает **та же самая** политика.

Экспортируется стратегия с наименьшим сожалением.


In [9]:
from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

best_name = min(results, key=lambda n: results[n][0].regret_per_pull(ARM_PROBABILITIES))
best_algo, _ = results[best_name]
print(f"экспортируется стратегия с наименьшим сожалением: {best_name}\n")

# Наблюдения из реальной среды: требование 10.5 говорит именно о реальном
# распределении, а не о случайном шуме.
collected, obs_now = [], vec.reset()
while sum(len(c) for c in collected) < 64:
    collected.append(obs_now[0].copy())
    obs_now = vec.step(best_algo.greedy_action(vec.num_envs)[:, None].astype(np.int32)).obs
sample = [np.concatenate(collected, axis=0)[:64]]

spec = ActionSpecLite(discrete_branches=(NUM_ARMS,))
policy = best_algo.policy_module()
run = create_run_dir(ENV_ID, best_name, SEED)
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
              "Models" / f"{ENV_ID}_{best_name}.onnx")
model_dest.parent.mkdir(parents=True, exist_ok=True)
model_dest.write_bytes(onnx_path.read_bytes())
print(f"\nмодель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")

vec.close()


экспортируется стратегия с наименьшим сожалением: thompson



OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    тип выхода deterministic_discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа discrete_action_output_shape — ожидалось [5.0], получено [5.0]
OK    числовой паритет: deterministic_discrete_actions — max|ONNX-PyTorch| = 0.000e+00, допуск 1e-04
OK    диапазон дискретной ветки 0 — значения [4, 4], допустимо целые [0, 5)
ИТОГ: ПРОЙДЕНО (13/13)

модель скопирована в проект Unity: unity\MLAgentsLab\Assets\Envs\E00_Bandit\Models\E00_Bandit_thompson.onnx


## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E00_Bandit__ucb.yaml --python-reward 0.80
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Порог — 0.8 от неё.

Оговорка про шум. Награда бандита бернуллиевская, и 20 эпизодов — это
20 нажатий: стандартная ошибка среднего около 0.09. Отношение Unity/Python
на такой выборке гуляет; устойчивый вывод даёт только исход «выбрана рука 4»,
который виден в `outcomes` (`Optimal` против `Suboptimal`).

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.8f1.
2. Открыть `Assets/Envs/E00_Bandit/Scenes/E00_Bandit.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента: `Model` = импортированная модель, `Behavior Type` = `Inference Only`,
   галочка `Deterministic Inference`.
5. Play: правый рычаг (рука 4) быстро «вырастает» — его высота равна доле
   нажатий. Остальные рычаги остаются низкими.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, клавиши **1…5** выбирают руку.
За несколько десятков нажатий видно, что правые руки платят чаще.
